In [1]:
#importing the libraries
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
# from tensorflow.keras.regularizers import L2
from tensorflow import keras 
from openpyxl import Workbook

%matplotlib inline

# 1. LOADING THE DATASET

In [2]:
df = pd.read_excel(r"C:\Users\GEN-X\Downloads\diabetes dataset.xlsx")
df.head()

,id,chol,stab.glu,hdl,ratio,glyhb,location,age,gender,height,weight,frame,bp.1s,bp.1d,bp.2s,bp.2d,waist,hip,time.ppn
0,1000,203.0,82.0,56.0,3.6,4.31,Buckingham,46.0,female,62.0,121.0,medium,118.0,59.0,NaN,NaN,29.0,38.0,720.0
1,1001,165.0,97.0,24.0,6.9,4.44,Buckingham,29.0,female,64.0,218.0,large,112.0,68.0,NaN,NaN,46.0,48.0,360.0
2,1002,228.0,92.0,37.0,6.2,4.64,Buckingham,58.0,female,61.0,256.0,large,190.0,92.0,185.0,92.0,49.0,57.0,180.0
3,1003,78.0,93.0,12.0,6.5,4.63,Buckingham,67.0,male,67.0,119.0,large,110.0,50.0,NaN,NaN,33.0,38.0,480.0
4,1005,249.0,90.0,28.0,8.9,7.72,Buckingham,64.0,male,68.0,183.0,medium,138.0,80.0,NaN,NaN,44.0,41.0,300.0


In [3]:
# dataset shape
df.shape

(4416, 19)

# 2. DATA PREPROCESSING

In [4]:
# checking for data infor
data = df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4416 entries, 0 to 4415
Data columns (total 19 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        4416 non-null   int64  
 1   chol      4414 non-null   float64
 2   stab.glu  4416 non-null   float64
 3   hdl       4414 non-null   float64
 4   ratio     4414 non-null   float64
 5   glyhb     4390 non-null   float64
 6   location  4416 non-null   object 
 7   age       4416 non-null   float64
 8   gender    4416 non-null   object 
 9   height    4406 non-null   float64
 10  weight    4414 non-null   float64
 11  frame     4392 non-null   object 
 12  bp.1s     4406 non-null   float64
 13  bp.1d     4406 non-null   float64
 14  bp.2s     3892 non-null   float64
 15  bp.2d     3892 non-null   float64
 16  waist     4412 non-null   float64
 17  hip       4412 non-null   float64
 18  time.ppn  4410 non-null   float64
dtypes: float64(15), int64(1), object(3)
memory usage: 655.6+ KB


In [5]:
# checking missing value
nan_values = df.isna().sum()
missing_values = nan_values[nan_values > 0]
print('Below are the columns with missing values:\n',missing_values)

# percentage of each column with missing values
data = (missing_values/len(df))*100
print('\n The following is the percentage of missing values:\n',data)

Below are the columns with missing values:
 chol          2
hdl           2
ratio         2
glyhb        26
height       10
weight        2
frame        24
bp.1s        10
bp.1d        10
bp.2s       524
bp.2d       524
waist         4
hip           4
time.ppn      6
dtype: int64

 The following is the percentage of missing values:
 chol         0.045290
hdl          0.045290
ratio        0.045290
glyhb        0.588768
height       0.226449
weight       0.045290
frame        0.543478
bp.1s        0.226449
bp.1d        0.226449
bp.2s       11.865942
bp.2d       11.865942
waist        0.090580
hip          0.090580
time.ppn     0.135870
dtype: float64


In [6]:
# dropping nan values
data = data.dropna()
# dropping columns
#data = data.drop(columns = ['id'])
data.isna().sum()

0

In [7]:
# filling columns which have missing values
df['bp.2d'] = df['bp.2d'].fillna(df['bp.2d'].median()) 
df['bp.2s'] = df['bp.2s'].fillna(df['bp.2s'].median())

# dropping rows with nan values
data = df.dropna()

num_cols = [col for col in data.columns if data[col].dtype != "object"]
print('numeric col include:', num_cols)
cat_cols = [col for col in data.columns if data[col].dtype == "object"]
print('cat col include:', cat_cols)

numeric col include: ['id', 'chol', 'stab.glu', 'hdl', 'ratio', 'glyhb', 'age', 'height', 'weight', 'bp.1s', 'bp.1d', 'bp.2s', 'bp.2d', 'waist', 'hip', 'time.ppn']
cat col include: ['location', 'gender', 'frame']


In [8]:
# checking for duplicate values
data.duplicated().sum()

3

In [9]:
# removing duplicate values
data = data.drop_duplicates()
data.duplicated().sum()

0

In [10]:
# create a new column
#data = data.dropna(subset=['glyhb'])
data['diabetes'] = ((data['glyhb'] > 7) | (data['glyhb'] < 5)).astype(int)
data.head()

,id,chol,stab.glu,hdl,ratio,glyhb,location,age,gender,height,weight,frame,bp.1s,bp.1d,bp.2s,bp.2d,waist,hip,time.ppn,diabetes
0,1000,203.0,82.0,56.0,3.6,4.31,Buckingham,46.0,female,62.0,121.0,medium,118.0,59.0,152.38,92.52,29.0,38.0,720.0,1
1,1001,165.0,97.0,24.0,6.9,4.44,Buckingham,29.0,female,64.0,218.0,large,112.0,68.0,152.38,92.52,46.0,48.0,360.0,1
2,1002,228.0,92.0,37.0,6.2,4.64,Buckingham,58.0,female,61.0,256.0,large,190.0,92.0,185.00,92.00,49.0,57.0,180.0,1
3,1003,78.0,93.0,12.0,6.5,4.63,Buckingham,67.0,male,67.0,119.0,large,110.0,50.0,152.38,92.52,33.0,38.0,480.0,1
4,1005,249.0,90.0,28.0,8.9,7.72,Buckingham,64.0,male,68.0,183.0,medium,138.0,80.0,152.38,92.52,44.0,41.0,300.0,1


In [11]:
# dropping columns
data1 = data.drop(columns = ['id','gender','location', 'frame','glyhb'])
data1.head()

,chol,stab.glu,hdl,ratio,age,height,weight,bp.1s,bp.1d,bp.2s,bp.2d,waist,hip,time.ppn,diabetes
0,203.0,82.0,56.0,3.6,46.0,62.0,121.0,118.0,59.0,152.38,92.52,29.0,38.0,720.0,1
1,165.0,97.0,24.0,6.9,29.0,64.0,218.0,112.0,68.0,152.38,92.52,46.0,48.0,360.0,1
2,228.0,92.0,37.0,6.2,58.0,61.0,256.0,190.0,92.0,185.00,92.00,49.0,57.0,180.0,1
3,78.0,93.0,12.0,6.5,67.0,67.0,119.0,110.0,50.0,152.38,92.52,33.0,38.0,480.0,1
4,249.0,90.0,28.0,8.9,64.0,68.0,183.0,138.0,80.0,152.38,92.52,44.0,41.0,300.0,1


In [12]:
# dropping columns which missing values are greater than 10%
data = data1.drop(columns = ['bp.2s','bp.2d'])

num_cols = [col for col in data.columns if data[col].dtype != "object"]
print('numeric col include:', num_cols)
cat_cols = [col for col in data.columns if data[col].dtype == "object"]
print('cat col include:', cat_cols)

numeric col include: ['chol', 'stab.glu', 'hdl', 'ratio', 'age', 'height', 'weight', 'bp.1s', 'bp.1d', 'waist', 'hip', 'time.ppn', 'diabetes']
cat col include: []


In [13]:
# Feature / target split 

X = data.drop(columns=['diabetes']).values.astype(np.float32)
y = data['diabetes'].values.astype(np.float32)
feature_names = data.drop(columns=['diabetes']).columns.tolist()

In [14]:
# Scale 
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y, test_size=0.2, random_state=42, stratify=y
)

print(f"  Samples  : {len(df)}  (train={len(X_train)}, test={len(X_test)})")
print(f"  Features : {X.shape[1]}")
print(f"  Diabetic : {int(y.sum())} / Non-Diabetic : {int((y==0).sum())}")

  Samples  : 4416  (train=3471, test=868)
  Features : 12
  Diabetic : 521 / Non-Diabetic : 3818


In [15]:
#  ACTIVATIONS & LOSSES

def relu(z):         return np.maximum(0, z)
def relu_d(z):       return (z > 0).astype(np.float32)
def sigmoid(z):      return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def bce_loss(y_true, y_pred):
    p = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

def layer_norm(x, eps=1e-6):
    mu  = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1,  keepdims=True) + eps
    return (x - mu) / std


In [16]:
# defining the model

class DeepANN:
    def __init__(self, input_dim, hidden=[256, 128, 64, 32],
                 lr=0.001, dropout=0.3, l2=1e-4, seed=42):
        np.random.seed(seed)
        self.lr      = lr
        self.dropout = dropout
        self.l2      = l2
        self.loss_hist = []
        self.val_hist  = []

        sizes = [input_dim] + hidden + [1]
        self.W, self.b = [], []
        self.mW, self.vW = [], []   # Adam moments
        self.mb, self.vb = [], []
        self.t = 0                  # Adam timestep

        for i in range(len(sizes) - 1):
            w = np.random.randn(sizes[i], sizes[i+1]).astype(np.float32) \
                * np.sqrt(2.0 / sizes[i])
            b = np.zeros((1, sizes[i+1]), dtype=np.float32)
            self.W.append(w); self.b.append(b)
            self.mW.append(np.zeros_like(w)); self.vW.append(np.zeros_like(w))
            self.mb.append(np.zeros_like(b)); self.vb.append(np.zeros_like(b))
    
# adding the optimizer layer
    def _adam(self, dW, db, i, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        self.mW[i] = beta1 * self.mW[i] + (1 - beta1) * dW
        self.vW[i] = beta2 * self.vW[i] + (1 - beta2) * dW**2
        mW_hat = self.mW[i] / (1 - beta1**self.t)
        vW_hat = self.vW[i] / (1 - beta2**self.t)
        self.W[i] -= self.lr * (mW_hat / (np.sqrt(vW_hat) + eps) + self.l2 * self.W[i])

        self.mb[i] = beta1 * self.mb[i] + (1 - beta1) * db
        self.vb[i] = beta2 * self.vb[i] + (1 - beta2) * db**2
        mb_hat = self.mb[i] / (1 - beta1**self.t)
        vb_hat = self.vb[i] / (1 - beta2**self.t)
        self.b[i] -= self.lr * mb_hat / (np.sqrt(vb_hat) + eps)
        
# doing forward pass
    def forward(self, X, training=True):
        self.acts  = [X]
        self.zs    = []
        self.masks = []

        for i, (w, b) in enumerate(zip(self.W, self.b)):
            z = self.acts[-1] @ w + b
            self.zs.append(z)

            if i < len(self.W) - 1:          # hidden → ReLU + Dropout
                a = relu(z)
                if training:
                    m = (np.random.rand(*a.shape) > self.dropout).astype(np.float32)
                    a = a * m / (1 - self.dropout)
                else:
                    m = np.ones_like(a)
                self.masks.append(m)
            else:                             # output → Sigmoid
                a = sigmoid(z)
                self.masks.append(np.ones_like(a))

            self.acts.append(a)

        return self.acts[-1]
        
# doing backwards pass
    def backward(self, y):
        m   = y.shape[0]
        y   = y.reshape(-1, 1)
        dA  = (self.acts[-1] - y) / (self.acts[-1] * (1 - self.acts[-1]) + 1e-8) \
              * self.acts[-1] * (1 - self.acts[-1])

        for i in reversed(range(len(self.W))):
            dZ = dA if i == len(self.W) - 1 else dA * relu_d(self.zs[i]) * self.masks[i]
            dW = self.acts[i].T @ dZ / m
            db = dZ.mean(axis=0, keepdims=True)
            if i > 0:
                dA = dZ @ self.W[i].T
            self._adam(dW, db, i)

    def fit(self, X, y, X_val=None, y_val=None,
            epochs=200, batch=32, verbose=True):
        for ep in range(1, epochs + 1):
            idx = np.random.permutation(len(X))
            for s in range(0, len(X), batch):
                Xb, yb = X[idx[s:s+batch]], y[idx[s:s+batch]]
                self.forward(Xb, training=True)
                self.backward(yb)

            out  = self.forward(X, training=False).flatten()
            loss = bce_loss(y, out)
            self.loss_hist.append(loss)

            if X_val is not None:
                vout = self.forward(X_val, training=False).flatten()
                self.val_hist.append(bce_loss(y_val, vout))

            if verbose and ep % 40 == 0:
                acc = accuracy_score(y, (out > 0.5).astype(int))
                vacc = accuracy_score(y_val, (vout > 0.5).astype(int)) if X_val is not None else 0
                print(f"    Epoch {ep:>3}/{epochs}  loss={loss:.4f}  acc={acc:.4f}  val_acc={vacc:.4f}")

    def predict_proba(self, X):
        return self.forward(X, training=False).flatten()

    def predict(self, X, thr=0.5):
        return (self.predict_proba(X) > thr).astype(int)


In [17]:
# compiling the model
ann = DeepANN(input_dim=X_train.shape[1], hidden=[256, 128, 64, 32],
              lr=0.001, dropout=0.3)
ann.fit(X_train, y_train, X_val=X_test, y_val=y_test,
        epochs=200, batch=32, verbose=True)

ann_prob = ann.predict_proba(X_test)
ann_pred = ann.predict(X_test)

KeyboardInterrupt: 

In [ ]:
print(f"\n Acc={accuracy_score(y_test,ann_pred):.4f}  AUC={roc_auc_score(y_test,ann_prob):.4f}   F1={f1_score(y_test, ann_pred):.4f}")

In [ ]:
# building a tabtransformer
class MultiHeadAttention:
    """
    Multi-head self-attention for tabular data.
    Input  : (batch, num_features, d_model)
    Output : (batch, num_features, d_model)
    """
    def __init__(self, d_model, num_heads, seed=42):
        np.random.seed(seed)
        assert d_model % num_heads == 0
        self.h   = num_heads
        self.dk  = d_model // num_heads
        self.dm  = d_model
        s = np.sqrt(d_model)

        # Q, K, V projections + output
        self.Wq = np.random.randn(d_model, d_model).astype(np.float32) / s
        self.Wk = np.random.randn(d_model, d_model).astype(np.float32) / s
        self.Wv = np.random.randn(d_model, d_model).astype(np.float32) / s
        self.Wo = np.random.randn(d_model, d_model).astype(np.float32) / s

        # Adam moments
        for name in ['Wq','Wk','Wv','Wo']:
            setattr(self, f'm{name}', np.zeros_like(getattr(self, name)))
            setattr(self, f'v{name}', np.zeros_like(getattr(self, name)))
        self.t = 0

    def _split_heads(self, x):
        # x: (B, S, D) → (B, h, S, dk)
        B, S, _ = x.shape
        return x.reshape(B, S, self.h, self.dk).transpose(0, 2, 1, 3)

    def _merge_heads(self, x):
        # x: (B, h, S, dk) → (B, S, D)
        B, h, S, dk = x.shape
        return x.transpose(0, 2, 1, 3).reshape(B, S, h * dk)

    def forward(self, x):
        self._x = x
        B, S, D = x.shape

        # Linear projections
        self._Q = x @ self.Wq  # (B,S,D)
        self._K = x @ self.Wk
        self._V = x @ self.Wv

        Q = self._split_heads(self._Q)   # (B,h,S,dk)
        K = self._split_heads(self._K)
        V = self._split_heads(self._V)

        # Scaled dot-product attention
        scores = Q @ K.transpose(0,1,3,2) / np.sqrt(self.dk)  # (B,h,S,S)
        self._A = softmax(scores)                               # attention weights
        context = self._A @ V                                   # (B,h,S,dk)

        # Merge heads + output projection
        out = self._merge_heads(context) @ self.Wo              # (B,S,D)
        self._out = out
        return out

    def backward(self, dout, lr, l2=1e-4, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        B, S, D = dout.shape

        # Gradient through Wo
        ctx = self._merge_heads(self._A @ self._split_heads(self._V))
        dWo = ctx.reshape(-1, D).T @ dout.reshape(-1, D) / B
        dWo += l2 * self.Wo
        dctx = dout @ self.Wo.T

        # Update Wo with Adam
        self.mWo = beta1*self.mWo + (1-beta1)*dWo
        self.vWo = beta2*self.vWo + (1-beta2)*dWo**2
        self.Wo -= lr * (self.mWo/(1-beta1**self.t)) / (np.sqrt(self.vWo/(1-beta2**self.t))+eps)

        # Simplified gradient through Q,K,V projections
        dx = dctx @ (self.Wq + self.Wk + self.Wv).T / 3
        for Wname, grad in [('Wq', self._x.reshape(-1,D).T @ dctx.reshape(-1,D) / B),
                             ('Wk', self._x.reshape(-1,D).T @ dctx.reshape(-1,D) / B),
                             ('Wv', self._x.reshape(-1,D).T @ dctx.reshape(-1,D) / B)]:
            W = getattr(self, Wname)
            mW = getattr(self, f'm{Wname}')
            vW = getattr(self, f'v{Wname}')
            g = grad + l2 * W
            mW = beta1*mW + (1-beta1)*g
            vW = beta2*vW + (1-beta2)*g**2
            setattr(self, f'm{Wname}', mW)
            setattr(self, f'v{Wname}', vW)
            setattr(self, Wname, W - lr*(mW/(1-beta1**self.t))/(np.sqrt(vW/(1-beta2**self.t))+eps))

        return dx


class TransformerBlock:
    """
    Full Transformer block:
    x → MHA → Add & Norm → FFN → Add & Norm
    """
    def __init__(self, d_model, num_heads, ff_dim, seed=42):
        self.attn = MultiHeadAttention(d_model, num_heads, seed)
        s = np.sqrt(d_model)
        self.W1 = np.random.randn(d_model, ff_dim).astype(np.float32) / s
        self.b1 = np.zeros((1, 1, ff_dim), dtype=np.float32)
        self.W2 = np.random.randn(ff_dim, d_model).astype(np.float32) / np.sqrt(ff_dim)
        self.b2 = np.zeros((1, 1, d_model), dtype=np.float32)
        # Adam
        for n in ['W1','b1','W2','b2']:
            setattr(self, f'm{n}', np.zeros_like(getattr(self, n)))
            setattr(self, f'v{n}', np.zeros_like(getattr(self, n)))
        self.t = 0

    def forward(self, x):
        self._x = x
        attn_out = self.attn.forward(x)
        self._ln1 = layer_norm(x + attn_out)            # Add & Norm

        ffn = relu(self._ln1 @ self.W1 + self.b1)       # FFN
        self._ffn_pre = ffn
        ffn_out = ffn @ self.W2 + self.b2
        out = layer_norm(self._ln1 + ffn_out)           # Add & Norm
        self._out = out
        return out

    def backward(self, dout, lr, l2=1e-4, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        B, S, D = dout.shape

        # FFN gradients
        dW2 = self._ffn_pre.reshape(-1, self._ffn_pre.shape[-1]).T \
              @ dout.reshape(-1, D) / B + l2 * self.W2
        db2 = dout.mean(axis=(0,1), keepdims=True)
        dffn = dout @ self.W2.T * relu_d(self._ln1 @ self.W1 + self.b1)
        dW1 = self._ln1.reshape(-1, D).T @ dffn.reshape(-1, dffn.shape[-1]) / B + l2 * self.W1
        db1 = dffn.mean(axis=(0,1), keepdims=True)

        # Adam updates for FFN weights
        for Wn, g in [('W1',dW1),('b1',db1),('W2',dW2),('b2',db2)]:
            W  = getattr(self, Wn)
            mW = getattr(self, f'm{Wn}')
            vW = getattr(self, f'v{Wn}')
            mW = beta1*mW + (1-beta1)*g
            vW = beta2*vW + (1-beta2)*g**2
            setattr(self, f'm{Wn}', mW)
            setattr(self, f'v{Wn}', vW)
            setattr(self, Wn, W - lr*(mW/(1-beta1**self.t))/(np.sqrt(vW/(1-beta2**self.t))+eps))

        # Attention backward
        dffn_reshaped = dffn.reshape(B, S, D) if dffn.size == B*S*D else dout
        dattn = self.attn.backward(dout + dffn_reshaped, lr, l2)
        return dattn + dout


class TabTransformer:
    """
    TabTransformer for tabular data:
      1. Linear embedding per feature  → d_model
      2. N Transformer blocks
      3. Flatten → MLP → sigmoid output
    """
    def __init__(self, num_features, d_model=32, num_heads=4,
                 ff_dim=64, num_layers=2, mlp_hidden=64,
                 lr=0.001, dropout=0.2, seed=42):
        np.random.seed(seed)
        self.lr      = lr
        self.dropout = dropout
        self.d       = d_model
        self.nf      = num_features
        self.loss_hist = []
        self.val_hist  = []

        # Per-feature embedding (1 → d_model)
        self.embed_W = np.random.randn(num_features, d_model).astype(np.float32) \
                       * 0.1
        self.embed_b = np.zeros((num_features, d_model), dtype=np.float32)
        self.me_W    = np.zeros_like(self.embed_W)
        self.ve_W    = np.zeros_like(self.embed_W)
        self.t       = 0

        # Transformer blocks
        self.blocks = [TransformerBlock(d_model, num_heads, ff_dim, seed+i)
                       for i in range(num_layers)]

        # MLP head
        flat = num_features * d_model
        self.Wm1 = np.random.randn(flat, mlp_hidden).astype(np.float32) / np.sqrt(flat)
        self.bm1 = np.zeros((1, mlp_hidden), dtype=np.float32)
        self.Wm2 = np.random.randn(mlp_hidden, 1).astype(np.float32) / np.sqrt(mlp_hidden)
        self.bm2 = np.zeros((1, 1), dtype=np.float32)
        for n in ['Wm1','bm1','Wm2','bm2']:
            setattr(self, f'mm{n}', np.zeros_like(getattr(self, n)))
            setattr(self, f'vm{n}', np.zeros_like(getattr(self, n)))

    def _embed(self, X):
        # X: (B, F) → (B, F, d_model)  via broadcasting
        return X[:, :, None] * self.embed_W[None] + self.embed_b[None]

    def forward(self, X, training=True):
        self._X = X
        emb = self._embed(X)           # (B, F, d)
        self._emb = emb

        out = emb
        for blk in self.blocks:
            out = blk.forward(out)
        self._blk_out = out

        # Flatten → MLP
        flat = out.reshape(len(X), -1)
        self._flat = flat

        h = relu(flat @ self.Wm1 + self.bm1)
        if training:
            mask = (np.random.rand(*h.shape) > self.dropout).astype(np.float32)
            h = h * mask / (1 - self.dropout)
        self._h   = h
        logit = h @ self.Wm2 + self.bm2
        prob  = sigmoid(logit).flatten()
        self._prob = prob
        return prob

    def _adam_update(self, param_name, grad, lr, beta1=0.9, beta2=0.999, eps=1e-8, l2=1e-4):
        self.t += 1
        W  = getattr(self, param_name)
        mW = getattr(self, f'mm{param_name}')
        vW = getattr(self, f'vm{param_name}')
        g  = grad + l2 * W
        mW = beta1*mW + (1-beta1)*g
        vW = beta2*vW + (1-beta2)*g**2
        setattr(self, f'mm{param_name}', mW)
        setattr(self, f'vm{param_name}', vW)
        setattr(self, param_name, W - lr*(mW/(1-beta1**self.t))/(np.sqrt(vW/(1-beta2**self.t))+eps))

    def backward(self, y):
        B  = len(y)
        dp = (self._prob - y) / B

        # MLP head gradients
        dWm2 = self._h.T @ dp.reshape(-1,1)
        dbm2 = dp.reshape(1,-1).mean(keepdims=True)
        dh   = dp.reshape(-1,1) @ self.Wm2.T * relu_d(self._flat @ self.Wm1 + self.bm1)
        dWm1 = self._flat.T @ dh / B
        dbm1 = dh.mean(axis=0, keepdims=True)

        self._adam_update('Wm2', dWm2, self.lr)
        self._adam_update('bm2', dbm2, self.lr)
        self._adam_update('Wm1', dWm1, self.lr)
        self._adam_update('bm1', dbm1, self.lr)

        # Transformer backward
        dflat = dh @ self.Wm1.T
        dblk  = dflat.reshape(B, self.nf, self.d)
        for blk in reversed(self.blocks):
            dblk = blk.backward(dblk, self.lr)

        # Embedding backward
        demb_W = (self._X[:, :, None] * dblk).mean(axis=0)
        self.me_W = 0.9*self.me_W + 0.1*demb_W
        self.ve_W = 0.999*self.ve_W + 0.001*demb_W**2
        self.embed_W -= self.lr * self.me_W / (np.sqrt(self.ve_W) + 1e-8)

    def fit(self, X, y, X_val=None, y_val=None,
            epochs=200, batch=32, verbose=True):
        for ep in range(1, epochs + 1):
            idx = np.random.permutation(len(X))
            for s in range(0, len(X), batch):
                Xb = X[idx[s:s+batch]]
                yb = y[idx[s:s+batch]]
                self.forward(Xb, training=True)
                self.backward(yb)

            out  = self.forward(X, training=False)
            loss = bce_loss(y, out)
            self.loss_hist.append(loss)

            if X_val is not None:
                vout = self.forward(X_val, training=False)
                self.val_hist.append(bce_loss(y_val, vout))

            if verbose and ep % 40 == 0:
                acc  = accuracy_score(y,     (out  > 0.5).astype(int))
                vacc = accuracy_score(y_val, (vout > 0.5).astype(int)) if X_val is not None else 0
                print(f"    Epoch {ep:>3}/{epochs}  loss={loss:.4f}  acc={acc:.4f}  val_acc={vacc:.4f}")

    def predict_proba(self, X):
        return self.forward(X, training=False)

    def predict(self, X, thr=0.5):
        return (self.predict_proba(X) > thr).astype(int)

    def get_attention_weights(self, X):
        """Returns attention weights from last transformer block for interpretability."""
        self.forward(X[:1], training=False)
        return self.blocks[-1].attn._A[0]  

In [ ]:
# compiling the model
tab = TabTransformer(
    num_features=X_train.shape[1],
    d_model=32, num_heads=4, ff_dim=64,
    num_layers=2, mlp_hidden=64,
    lr=0.001, dropout=0.2
)
tab.fit(X_train, y_train, X_val=X_test, y_val=y_test,
        epochs=200, batch=32, verbose=True)

tab_prob = tab.predict_proba(X_test)
tab_pred = tab.predict(X_test)

In [ ]:
print(f"TabTransformer → Acc={accuracy_score(y_test,tab_pred):.4f}  AUC={roc_auc_score(y_test,tab_prob):.4f}    F1={f1_score(y_test, tab_pred):.4f}")

In [ ]:
class HybridANNTransformer:
    """
    Combines a Deep ANN (MLP) with a TabTransformer.
    """
    def __init__(self, num_features, d_model=32, num_heads=4, ff_dim=64,
                 num_layers=2, mlp_hidden=[128, 64], ann_hidden=[256, 128],
                 final_hidden=64, lr=0.001, dropout=0.2, seed=42):
        np.random.seed(seed)
        self.lr = lr
        self.dropout = dropout
        self.d = d_model
        self.num_features = num_features
        self.loss_hist = []
        self.val_hist = []

        # ---------- Transformer branch ----------
        # per‑feature linear embedding
        self.embed_W = np.random.randn(num_features, d_model).astype(np.float32) * 0.1
        self.embed_b = np.zeros((num_features, d_model), dtype=np.float32)
        self.me_W = np.zeros_like(self.embed_W)
        self.ve_W = np.zeros_like(self.embed_W)
        # transformer blocks
        self.blocks = [TransformerBlock(d_model, num_heads, ff_dim, seed+i)
                       for i in range(num_layers)]
        # transformer output projection to final hidden size
        trans_out_dim = num_features * d_model
        self.trans_proj_W = np.random.randn(trans_out_dim, final_hidden).astype(np.float32) / np.sqrt(trans_out_dim)
        self.trans_proj_b = np.zeros((1, final_hidden), dtype=np.float32)

        # ---------- ANN branch ----------
        sizes = [num_features] + ann_hidden + [final_hidden]
        self.W_ann, self.b_ann = [], []
        self.mW_ann, self.vW_ann = [], []
        self.mb_ann, self.vb_ann = [], []
        for i in range(len(sizes)-1):
            w = np.random.randn(sizes[i], sizes[i+1]).astype(np.float32) * np.sqrt(2.0/sizes[i])
            b = np.zeros((1, sizes[i+1]), dtype=np.float32)
            self.W_ann.append(w)
            self.b_ann.append(b)
            self.mW_ann.append(np.zeros_like(w))
            self.vW_ann.append(np.zeros_like(w))
            self.mb_ann.append(np.zeros_like(b))
            self.vb_ann.append(np.zeros_like(b))

        # ---------- Final classifier ----------
        self.final_W = np.random.randn(2*final_hidden, 1).astype(np.float32) / np.sqrt(2*final_hidden)
        self.final_b = np.zeros((1, 1), dtype=np.float32)

        # Adam parameters for all trainable weights
        self.t = 0
        # Store references for all parameters
        self._params = []
        # We'll use a unified Adam update later; for simplicity, we update each layer in backward.
        # For brevity, we'll use the same Adam helper inside backward.

    def _embed(self, X):
        """X: (B, F) -> (B, F, d_model)"""
        return X[:, :, None] * self.embed_W[None] + self.embed_b[None]

    def _ann_forward(self, X, training):
        """Forward pass of the ANN branch, returns final hidden representation (B, final_hidden)."""
        h = X
        self._ann_acts = [h]
        self._ann_zs = []
        self._ann_masks = []
        for i, (W, b) in enumerate(zip(self.W_ann, self.b_ann)):
            z = h @ W + b
            self._ann_zs.append(z)
            if i < len(self.W_ann) - 1:
                a = relu(z)
                if training:
                    mask = (np.random.rand(*a.shape) > self.dropout).astype(np.float32)
                    a = a * mask / (1 - self.dropout)
                else:
                    mask = np.ones_like(a)
                self._ann_masks.append(mask)
                h = a
                self._ann_acts.append(a)
            else:
                # last layer: no activation, just linear output
                self._ann_acts.append(z)
                h = z
        return h

    def _transformer_forward(self, X, training):
        """Forward pass of the transformer branch, returns final hidden representation (B, final_hidden)."""
        emb = self._embed(X)
        self._trans_emb = emb
        out = emb
        for blk in self.blocks:
            out = blk.forward(out)
        self._trans_out = out
        flat = out.reshape(len(X), -1)
        self._trans_flat = flat
        h = relu(flat @ self.trans_proj_W + self.trans_proj_b)
        if training:
            mask = (np.random.rand(*h.shape) > self.dropout).astype(np.float32)
            h = h * mask / (1 - self.dropout)
        self._trans_h = h
        return h

    def forward(self, X, training=True):
        self._X = X
        # ANN branch
        ann_out = self._ann_forward(X, training)
        # Transformer branch
        trans_out = self._transformer_forward(X, training)
        # Concatenate
        combined = np.concatenate([ann_out, trans_out], axis=1)
        # Final classification
        logit = combined @ self.final_W + self.final_b
        prob = sigmoid(logit).flatten()
        self._prob = prob
        self._combined = combined
        return prob

    def _adam_update(self, param, grad, lr, beta1=0.9, beta2=0.999, eps=1e-8, l2=1e-4):
        """Generic Adam update for a given parameter and its moments."""
        self.t += 1
        # This is simplified; in practice we store moments per parameter.
        # For brevity, we'll use a naive SGD update in the backward methods.
        # Instead, we implement a per‑layer Adam update in each component.
        pass  # Actually we will use the specific Adam code from the original classes.

    def backward(self, y):
        B = len(y)
        dp = (self._prob - y) / B

        # Gradient through final layer
        dfinal = dp.reshape(-1, 1)
        dcombined = dfinal @ self.final_W.T
        dW_final = self._combined.T @ dfinal
        db_final = dfinal.mean(axis=0, keepdims=True)

        # Update final layer with Adam
        self.t += 1
        # For simplicity, we use a separate Adam for each parameter.
        # We'll mimic the style from the original code.
        # We'll create attributes for moments (here we do it on the fly)
        if not hasattr(self, 'm_final_W'):
            self.m_final_W = np.zeros_like(self.final_W)
            self.v_final_W = np.zeros_like(self.final_W)
            self.m_final_b = np.zeros_like(self.final_b)
            self.v_final_b = np.zeros_like(self.final_b)
        # Update W_final
        self.m_final_W = 0.9 * self.m_final_W + 0.1 * dW_final
        self.v_final_W = 0.999 * self.v_final_W + 0.001 * dW_final**2
        self.final_W -= self.lr * self.m_final_W / (np.sqrt(self.v_final_W) + 1e-8)
        # Update b_final
        self.m_final_b = 0.9 * self.m_final_b + 0.1 * db_final
        self.v_final_b = 0.999 * self.v_final_b + 0.001 * db_final**2
        self.final_b -= self.lr * self.m_final_b / (np.sqrt(self.v_final_b) + 1e-8)

        # Split the combined gradient into ANN and Transformer branches
        d_ann, d_trans = np.split(dcombined, [self.W_ann[-1].shape[1]], axis=1)

        # ---------- ANN branch backward ----------
        # Backprop through ANN
        dA = d_ann
        for i in reversed(range(len(self.W_ann))):
            z = self._ann_zs[i]
            if i == len(self.W_ann) - 1:
                dZ = dA  # last layer is linear
            else:
                dZ = dA * relu_d(z) * self._ann_masks[i]
            dW = self._ann_acts[i].T @ dZ / B
            db = dZ.mean(axis=0, keepdims=True)
            # Adam update for this layer
            if not hasattr(self, f'mW_ann_{i}'):
                setattr(self, f'mW_ann_{i}', np.zeros_like(self.W_ann[i]))
                setattr(self, f'vW_ann_{i}', np.zeros_like(self.W_ann[i]))
                setattr(self, f'mb_ann_{i}', np.zeros_like(self.b_ann[i]))
                setattr(self, f'vb_ann_{i}', np.zeros_like(self.b_ann[i]))
            mW = getattr(self, f'mW_ann_{i}')
            vW = getattr(self, f'vW_ann_{i}')
            mW = 0.9 * mW + 0.1 * dW
            vW = 0.999 * vW + 0.001 * dW**2
            setattr(self, f'mW_ann_{i}', mW)
            setattr(self, f'vW_ann_{i}', vW)
            self.W_ann[i] -= self.lr * mW / (np.sqrt(vW) + 1e-8)

            mb = getattr(self, f'mb_ann_{i}')
            vb = getattr(self, f'vb_ann_{i}')
            mb = 0.9 * mb + 0.1 * db
            vb = 0.999 * vb + 0.001 * db**2
            setattr(self, f'mb_ann_{i}', mb)
            setattr(self, f'vb_ann_{i}', vb)
            self.b_ann[i] -= self.lr * mb / (np.sqrt(vb) + 1e-8)

            if i > 0:
                dA = dZ @ self.W_ann[i].T

        # ---------- Transformer branch backward ----------
        # Backprop through projection and transformer
        # d_trans is (B, final_hidden)
        d_proj = d_trans * relu_d(self._trans_flat @ self.trans_proj_W + self.trans_proj_b)
        dW_proj = self._trans_flat.T @ d_proj / B
        db_proj = d_proj.mean(axis=0, keepdims=True)
        # Update trans_proj
        if not hasattr(self, 'm_trans_proj_W'):
            self.m_trans_proj_W = np.zeros_like(self.trans_proj_W)
            self.v_trans_proj_W = np.zeros_like(self.trans_proj_W)
            self.m_trans_proj_b = np.zeros_like(self.trans_proj_b)
            self.v_trans_proj_b = np.zeros_like(self.trans_proj_b)
        self.m_trans_proj_W = 0.9 * self.m_trans_proj_W + 0.1 * dW_proj
        self.v_trans_proj_W = 0.999 * self.v_trans_proj_W + 0.001 * dW_proj**2
        self.trans_proj_W -= self.lr * self.m_trans_proj_W / (np.sqrt(self.v_trans_proj_W) + 1e-8)
        self.m_trans_proj_b = 0.9 * self.m_trans_proj_b + 0.1 * db_proj
        self.v_trans_proj_b = 0.999 * self.v_trans_proj_b + 0.001 * db_proj**2
        self.trans_proj_b -= self.lr * self.m_trans_proj_b / (np.sqrt(self.v_trans_proj_b) + 1e-8)

        dflat = d_proj @ self.trans_proj_W.T
        dblk = dflat.reshape(B, self.num_features, self.d)
        for blk in reversed(self.blocks):
            dblk = blk.backward(dblk, self.lr)

        # Embedding gradient
        demb_W = (self._X[:, :, None] * dblk).mean(axis=0)
        self.me_W = 0.9 * self.me_W + 0.1 * demb_W
        self.ve_W = 0.999 * self.ve_W + 0.001 * demb_W**2
        self.embed_W -= self.lr * self.me_W / (np.sqrt(self.ve_W) + 1e-8)

    def fit(self, X, y, X_val=None, y_val=None, epochs=200, batch=32, verbose=True):
        for ep in range(1, epochs + 1):
            idx = np.random.permutation(len(X))
            for s in range(0, len(X), batch):
                Xb = X[idx[s:s+batch]]
                yb = y[idx[s:s+batch]]
                self.forward(Xb, training=True)
                self.backward(yb)

            out = self.forward(X, training=False)
            loss = bce_loss(y, out)
            self.loss_hist.append(loss)

            if X_val is not None:
                vout = self.forward(X_val, training=False)
                self.val_hist.append(bce_loss(y_val, vout))

            if verbose and ep % 40 == 0:
                acc = accuracy_score(y, (out > 0.5).astype(int))
                vacc = accuracy_score(y_val, (vout > 0.5).astype(int)) if X_val is not None else 0
                print(f"    Epoch {ep:>3}/{epochs}  loss={loss:.4f}  acc={acc:.4f}  val_acc={vacc:.4f}")

    def predict_proba(self, X):
        return self.forward(X, training=False)

    def predict(self, X, thr=0.5):
        return (self.predict_proba(X) > thr).astype(int)

In [ ]:
hybrid = HybridANNTransformer(
    num_features=X_train.shape[1],
    d_model=32, num_heads=4, ff_dim=64,
    num_layers=2, ann_hidden=[256,128], final_hidden=64,
    lr=0.0001, dropout=0.2
)
hybrid.fit(X_train, y_train, X_val=X_test, y_val=y_test, epochs=200, batch=32, verbose=True)

hybrid_prob = hybrid.predict_proba(X_test)
hybrid_pred = hybrid.predict(X_test)

acc = accuracy_score(y_test, hybrid_pred)
auc = roc_auc_score(y_test, hybrid_prob)
f1 = f1_score(y_test, hybrid_pred)


In [ ]:
print(f"Hybrid Model → Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}")

In [ ]:
# results
print("\n  Hybrid Model — Cross Validation:")
def cv_evaluate(model_class, X, y, model_params, cv=5):
    """Perform stratified k-fold cross-validation and return mean AUC, std, and accuracy."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    auc_scores = []
    acc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]
        
        model = model_class(**model_params)
        model.fit(X_train_cv, y_train_cv, X_val=X_val_cv, y_val=y_val_cv, 
                  epochs=100, batch=32, verbose=False)
        
        y_pred_cv = model.predict_proba(X_val_cv)
        auc_scores.append(roc_auc_score(y_val_cv, y_pred_cv))
        acc_scores.append(accuracy_score(y_val_cv, (y_pred_cv > 0.5).astype(int)))
    
    return np.mean(auc_scores), np.std(auc_scores), np.mean(acc_scores)

hybrid_cv_auc, hybrid_cv_std, hybrid_cv_acc = cv_evaluate(
    HybridANNTransformer, X_sc, y,
    dict(num_features=X_sc.shape[1], d_model=32, num_heads=4, ff_dim=64,
         num_layers=2, ann_hidden=[256,128], final_hidden=64,
         lr=0.0001, dropout=0.2)
)
print(f"  → CV AUC = {hybrid_cv_auc:.4f} ± {hybrid_cv_std:.4f}  |  CV Acc = {hybrid_cv_acc:.4f}")

# ANN Cross Validation
print("\n  Deep ANN — Cross Validation:")
ann_cv_auc, ann_cv_std, ann_cv_acc = cv_evaluate(
    DeepANN, X_sc, y,
    dict(input_dim=X_sc.shape[1], hidden=[256, 128, 64, 32],
         lr=0.001, dropout=0.3)
)
print(f"  → CV AUC = {ann_cv_auc:.4f} ± {ann_cv_std:.4f}  |  CV Acc = {ann_cv_acc:.4f}")

# TabTransformer Cross Validation
print("\n  TabTransformer — Cross Validation:")
tab_cv_auc, tab_cv_std, tab_cv_acc = cv_evaluate(
    TabTransformer, X_sc, y,
    dict(num_features=X_sc.shape[1], d_model=32, num_heads=4, ff_dim=64,
         num_layers=2, mlp_hidden=64, lr=0.001, dropout=0.2)
)
print(f"  → CV AUC = {tab_cv_auc:.4f} ± {tab_cv_std:.4f}  |  CV Acc = {tab_cv_acc:.4f}")



In [ ]:
# results
results = {
    "Deep ANN (5-layer)": {
        "y_pred": ann_pred, "y_prob": ann_prob,
        "accuracy": accuracy_score(y_test, ann_pred),
        "roc_auc":  roc_auc_score(y_test, ann_prob),
        "f1":       f1_score(y_test, ann_pred),
        "cv_auc":   ann_cv_auc, "cv_std": ann_cv_std,
        "loss":     ann.loss_hist, "val_loss": ann.val_hist,
    },
    "TabTransformer": {
        "y_pred": tab_pred, "y_prob": tab_prob,
        "accuracy": accuracy_score(y_test, tab_pred),
        "roc_auc":  roc_auc_score(y_test, tab_prob),
        "f1":       f1_score(y_test, tab_pred),
        "cv_auc":   tab_cv_auc, "cv_std": tab_cv_std,
        "loss":     tab.loss_hist, "val_loss": tab.val_hist,
    },
    "Hybrid Model": {
        "y_pred": hybrid_pred, "y_prob": hybrid_prob,
        "accuracy": accuracy_score(y_test, hybrid_pred),
        "roc_auc":  roc_auc_score(y_test, hybrid_prob),
        "f1":       f1_score(y_test, hybrid_pred),
        "cv_auc":   hybrid_cv_auc, "cv_std": hybrid_cv_std,
        "loss":     hybrid.loss_hist, "val_loss": hybrid.val_hist,
    },
}

print(f"\n  {'Model':<25} {'Acc':>7} {'AUC':>7} {'F1':>7} {'CV AUC':>14}")
print("  " + "-" * 65)
for name, v in results.items():
    print(f"  {name:<25} {v['accuracy']:>7.4f} {v['roc_auc']:>7.4f} {v['f1']:>7.4f}  {v['cv_auc']:.4f}±{v['cv_std']:.4f}")

# VISUALATION

In [ ]:
# Create figure
fig = plt.figure(figsize=(20, 16))

# Main title
fig.suptitle(
    "Diabetes Prediction — Model Evaluation",
    fontsize=15,
    fontweight='bold',
    y=0.99)
 
# Grid layout (3 rows × 3 columns)
gs = gridspec.GridSpec(
    3, 3,
    figure=fig,
    hspace=0.5,
    wspace=0.35)
 
# Custom colors for models
colors = {
    "Deep ANN (5-layer)": "#2196F3",
    "TabTransformer": "#E91E63",
    "Hybrid Model": "#4CAF50"
}

# Example subplot usage
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("Accuracy Comparison")

models = list(colors.keys())
values = [0.89, 0.91, 0.95]

ax1.bar(models, values, color=[colors[m] for m in models])
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1)

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 16))
fig.suptitle("Diabetes Prediction — Model Evaluation",
             fontsize=15, fontweight='bold', y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35)

colors = {"Deep ANN (5-layer)": "#2196F3", "TabTransformer": "#E91E63", "Hybrid Model": "#4CAF50"}

In [ ]:
# ROC Curves
ax1 = fig.add_subplot(gs[0, :2])

for name, v in results.items():
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])

    ax1.plot(
        fpr, tpr,
        label=f"{name} (AUC={v['roc_auc']:.3f})",
        color=colors[name],
        lw=2.5
    )

# Diagonal baseline
ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)

# Fill areas under curves
ax1.fill_between(
    *roc_curve(y_test, ann_prob)[:2],
    alpha=0.08,
    color="#2196F3"
)

ax1.fill_between(
    *roc_curve(y_test, tab_prob)[:2],
    alpha=0.08,
    color="#E91E63"
)

ax1.fill_between(
    *roc_curve(y_test, hybrid_prob)[:2],
    alpha=0.08,
    color="#4CAF50"
)

# Labels and styling
ax1.set_title("ROC Curves", fontweight='bold', fontsize=12)
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")

ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

In [ ]:
# ROC Curves 
ax1 = fig.add_subplot(gs[0, :2])
for name, v in results.items():
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])
    ax1.plot(fpr, tpr, label=f"{name} (AUC={v['roc_auc']:.3f})",
             color=colors[name], lw=2.5)
ax1.plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
ax1.fill_between(*roc_curve(y_test, ann_prob)[:2],
                 alpha=0.08, color="#2196F3")
ax1.fill_between(*roc_curve(y_test, tab_prob)[:2],
                 alpha=0.08, color="#E91E63")
ax1.fill_between(*roc_curve(y_test, hybrid_prob)[:2],
                 alpha=0.08, color="#4CAF50")
ax1.set_title("ROC Curves", fontweight='bold', fontsize=12)
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.legend(fontsize=10); ax1.grid(alpha=0.3)

In [ ]:
# Metric bar chart
ax2 = fig.add_subplot(gs[0, 2])
metrics = ['accuracy', 'roc_auc', 'f1']
x = np.arange(len(metrics))
w = 0.25
for i, (name, v) in enumerate(results.items()):
    ax2.bar(x + i*w, [v[m] for m in metrics], w,
            label=name, color=colors[name], alpha=0.85)
ax2.set_xticks(x + w); ax2.set_xticklabels(['Accuracy','AUC','F1'], fontsize=9)
ax2.set_ylim(0, 1.1); ax2.set_title("Metric Comparison", fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

In [ ]:
# Confusion Matrices 
for idx, (name, v) in enumerate(results.items()):
    ax = fig.add_subplot(gs[1, idx])
    cm = confusion_matrix(y_test, v['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No DM','Diabetic'],
                yticklabels=['No DM','Diabetic'],
                annot_kws={"size": 13})
    ax.set_title(f"Confusion Matrix\n{name}", fontweight='bold', fontsize=10)
    ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")

In [ ]:
# CV AUC comparison 
ax5 = fig.add_subplot(gs[1, 2])
names  = list(results.keys())
cv_auc = [v['cv_auc'] for v in results.values()]
cv_std = [v['cv_std'] for v in results.values()]
bars = ax5.bar(names, cv_auc, color=[colors[n] for n in names],
               yerr=cv_std, capsize=8, alpha=0.85, width=0.4)
ax5.set_ylim(0.5, 1.0); ax5.set_title("5-Fold CV AUC", fontweight='bold')
ax5.set_ylabel("Mean AUC"); ax5.grid(axis='y', alpha=0.3)
ax5.set_xticklabels(names, fontsize=8)
for bar, val in zip(bars, cv_auc):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.4f}", ha='center', fontsize=9, fontweight='bold')

In [ ]:
# Training & Validation Loss 
for idx, (name, v) in enumerate(results.items()):
    ax = fig.add_subplot(gs[2, idx])
    ax.plot(v['loss'],     label='Train Loss', color=colors[name], lw=2)
    ax.plot(v['val_loss'], label='Val Loss',   color=colors[name], lw=1.5, linestyle='--', alpha=0.7)
    ax.set_title(f"Training Curve\n{name}", fontweight='bold', fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.show()

In [ ]:
# DETAILED CLASSIFICATION REPORT

print("Classification Reports...")

for name, v in results.items():
    print(f"\n  ── {name} ──")
    print(classification_report(y_test, v['y_pred'],
          target_names=["No Diabetes","Diabetic"]))
